# Lifetime aligned to a lick-bout onset

This example uses a fixed bout definition rather than hand-picking a lick. Licks separated by no more than one second belong to the same bout. The largest bout beginning between 90 and 150 seconds is selected, and vendor lifetime is aligned to its first lick.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pynapple as nap
from fluopulse_analysis import process_aligned_session

In [ ]:
RUN = 1
DORIC_FILE = Path.home() / "Documents" / "SC81_260923" / f"260923_SC81_run{RUN}_0000.doric"
NIDAQ_FOLDER = Path.home() / "Documents" / "nidaq" / "SC81_260923"
NIDAQ_FILE = NIDAQ_FOLDER / f"SC81-260923-{RUN:03d}-nidaq.mat"
RUNNING_FILE = NIDAQ_FOLDER / f"SC81-260923-{RUN:03d}-running.mat"
session = process_aligned_session(
    DORIC_FILE,
    NIDAQ_FILE,
    running_path=RUNNING_FILE if RUNNING_FILE.is_file() else None,
)
data = session.to_pynapple()

In [ ]:
MAX_INTERLICK_GAP_S = 1.0
SEARCH_WINDOW_S = (90.0, 150.0)
MIN_LICKS = 5
lick_times = np.asarray(session.doric_licks.onset_times)
bout_starts = np.r_[0, np.flatnonzero(np.diff(lick_times) > MAX_INTERLICK_GAP_S) + 1]
bout_stops = np.r_[bout_starts[1:], lick_times.size]
bout_onsets = lick_times[bout_starts]
bout_ends = lick_times[bout_stops - 1]
bout_counts = bout_stops - bout_starts
candidates = np.flatnonzero(
    (bout_onsets >= SEARCH_WINDOW_S[0])
    & (bout_onsets <= SEARCH_WINDOW_S[1])
    & (bout_counts >= MIN_LICKS)
)
if candidates.size == 0:
    raise ValueError("No lick bout satisfies the selection criteria")
major_bout = candidates[np.argmax(bout_counts[candidates])]
bout_onset = bout_onsets[major_bout]
print(f"Bout onset: {bout_onset:.3f} s")
print(f"Bout end: {bout_ends[major_bout]:.3f} s")
print(f"Licks: {bout_counts[major_bout]}")

In [ ]:
bout_event = nap.Ts(
    t=np.array([bout_onset]),
    time_units="s",
    time_support=data["tau"].time_support,
)
tau_around_bout = nap.compute_perievent(
    data["tau"],
    bout_event,
    window=(-20, 60),
    time_unit="s",
)
relative_time = np.asarray(tau_around_bout.t)
tau_values = np.asarray(tau_around_bout).squeeze()
baseline_mask = (relative_time >= -10) & (relative_time < 0)
delta_tau = tau_values - np.nanmean(tau_values[baseline_mask])
relative_licks = lick_times - bout_onset
relative_licks = relative_licks[(relative_licks >= -20) & (relative_licks <= 60)]

In [ ]:
figure, axes = plt.subplots(
    2, 1, figsize=(11, 6), sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
    constrained_layout=True,
)
axes[0].plot(relative_time, delta_tau, color="tab:blue")
axes[0].axvline(0, color="tab:orange", linestyle="--")
axes[0].set_ylabel("Delta tau (ns)")
axes[0].set_title("Lifetime aligned to major lick-bout onset")
axes[1].eventplot(relative_licks, colors="black")
axes[1].axvline(0, color="tab:orange", linestyle="--")
axes[1].set(xlabel="Time from bout onset (s)", ylabel="Licks")
plt.show()

For comparisons across recordings, keep `MAX_INTERLICK_GAP_S`, `MIN_LICKS`, the selection rule, baseline, and peri-event window fixed. The explicit lifetime `time_support` prevents the zero-duration warning that Pynapple otherwise emits for a newly created single-event `Ts` object.